<a href="https://colab.research.google.com/github/Deva2013/airline-disruption-management-system/blob/main/Airline_Disruption_Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ── Environment Setup ──────────────────────────────────────────────────────
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Install Hugging Face Hub client
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login

# ── Authenticate with Hugging Face using the Colab secret ──────────────────
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

# ── Directory structure ──────────────────────────────────────────────────
# Everything lives on LOCAL Colab disk (/content) — fast, reliable, no
# Drive quota issues. This does NOT persist across sessions; that's
# expected. Finished "gold" outputs get pushed to Hugging Face Hub at the
# end of each stage instead of being written to Drive.

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Base directory: {BASE_DIR}')
print(f'HF repo target: {HF_REPO_ID}')

Environment ready.
Base directory: /content/airline-disruption
HF repo target: Dev123Hug456Face/airline-disruption-data


In [2]:
# ── Pull the verified BTS dataset from Hugging Face Hub ────────────────────
from huggingface_hub import hf_hub_download

bts_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_cleaned.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

print(f"Downloaded to: {bts_path}")

# Quick verification
df = pd.read_parquet(bts_path)
print(f"\nRows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"\nColumn list: {df.columns.tolist()}")

bts_cleaned.parquet: reconstructing file:   0%|          |  0.00B /  355MB            

bts_cleaned.parquet: downloading bytes:           |  0.00B            

Downloaded to: /content/airline-disruption/data/processed/bts_cleaned.parquet

Rows: 10,504,936
Columns: 41
Year range: 2022 - 2024

Column list: ['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Airline', 'FlightNumber', 'OperatingAirline', 'OperatingAirlineCode', 'Origin', 'OriginCityName', 'OriginState', 'Dest', 'DestCityName', 'DestState', 'CRSDepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'TaxiOut', 'TaxiIn', 'CRSArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'Cancelled', 'CancellationCode', 'Diverted', 'AirTime', 'Distance', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'ScheduledDepHour', 'Season', 'IsWeekend']


In [3]:
# ── Dataset overview ────────────────────────────────────────────────────
total     = len(df)
cancelled = df['Cancelled'].sum()
dep_del   = df['DepDel15'].sum()
arr_del   = df['ArrDel15'].sum()

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Flights       : {total:,}')
print(f'  Date range    : {df["FlightDate"].min()} → {df["FlightDate"].max()}')
print(f'  Airlines      : {df["Airline"].nunique()}')
print(f'  Cancellation  : {cancelled/total*100:.2f}%  ({cancelled:,})')
print(f'  Dep delay≥15m : {dep_del/total*100:.2f}%  ({dep_del:,})')
print(f'  Arr delay≥15m : {arr_del/total*100:.2f}%  ({arr_del:,})')
print()
print('Severity breakdown:')
print(df['SeverityTier'].value_counts().to_string())

  DATASET OVERVIEW
  Flights       : 10,504,936
  Date range    : 2022-01-01 00:00:00 → 2024-12-31 00:00:00
  Airlines      : 10
  Cancellation  : 1.81%  (190,145)
  Dep delay≥15m : 20.04%  (2,105,664)
  Arr delay≥15m : 20.45%  (2,148,534)

Severity breakdown:
SeverityTier
On Time        8166257
Minor          1140970
Significant     697812
Severe          309752
Cancelled       190145
